<a href="https://colab.research.google.com/github/egaeuni/2026-1-DL-Final-Project/blob/main/DL_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install optuna

In [ ]:
# 필요한 라이브러리
import os
import warnings
from datetime import timedelta

warnings.filterwarnings('ignore')

from google.colab import drive

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import random
import copy

import holidays # 공휴일

# 데이터 전처리
from torch.utils.data import DataLoader, Dataset, TensorDataset
from sklearn.preprocessing import StandardScaler, MinMaxScaler

# 딥러닝
import torch
import torch.nn as nn
import torch.nn.functional as F

# 하이퍼파라미터 최적화
import optuna
from sklearn.metrics import mean_absolute_percentage_error

In [ ]:
# 시드 고정
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed) # 멀티 GPU 사용할 경우
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# 가장 먼저 시드 고정 실행
seed_everything(42)

In [ ]:
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"현재 할당된 장치: {device}")

In [ ]:
# ---------------------------------------------------------
# 1. 데이터 로드
# ---------------------------------------------------------
drive.mount('/content/drive')

DIR_PATH = '/content/drive/MyDrive/DeepLearning_Data/Final_Project'
train_path = os.path.join(DIR_PATH, 'train.csv')
test_path = os.path.join(DIR_PATH, 'test.csv')

raw_train_df = pd.read_csv(train_path)
raw_test_df = pd.read_csv(test_path)

print(f"최초 로드 - Train: {raw_train_df.shape} | Test: {raw_test_df.shape}")

In [ ]:
def add_kma_warning_features(df):
    """
    기상청 공식 기준 폭염·한파 주의보/경보 파생변수 생성.
    create_initial_features() 내부에서 호출.

    생성 변수
    ---------
    [공통 보조]
    HeatIndex         : 기상청 기준 체감온도(더위) - 시간별
    LowTemp_prev1d    : 전날 최저기온 (일별 shift)
    LowTemp_drop_1d   : 전날 대비 최저기온 하강폭 (°C)
    Below_avg3        : 평년 평균기온보다 3°C 이상 낮은 날 여부
    MaxHeatIndex_daily: 일 최고 체감온도 (시간→일 집계)

    [한파 - 10~4월만 적용]
    Coldwave_Watch    : 한파 주의보 (조건 ① or ②)
    Coldwave_Warning  : 한파 경보  (조건 ① or ②)
    Coldwave_Level    : 0=없음 / 1=주의보 / 2=경보 (순서 변수)

    [폭염]
    Heatwave_Watch    : 폭염 주의보 (일최고체감온도 33°C 이상 2일 이상)
    Heatwave_Warning  : 폭염 경보  (일최고체감온도 35°C 이상 2일 이상)
    Heatwave_Level    : 0=없음 / 1=주의보 / 2=경보 (순서 변수)

    [연속성]
    Coldwave_streak   : 한파 특보(주의보 이상) 연속 지속 시간 수
    Heatwave_streak   : 폭염 특보(주의보 이상) 연속 지속 시간 수
    """
    df = df.copy()

    # ── 1. 체감온도(Heat Index) 계산 ─────────────────────────────────────
    # 기상청 기준: 기온 27°C 이상 & 상대습도 40% 이상일 때만 수식 적용
    # 그 외: 실제 기온(Temp) 그대로 사용
    T, H, V = df['Temp'], df['Humi'], df['Wind']
    heat_idx_raw = (-0.2442 + 0.55399 * T + 0.45535 * H
                    - 0.0022 * H**2 + 0.00278 * T * H)
    df['HeatIndex'] = np.where((T >= 27) & (H >= 40), heat_idx_raw, T)

    # ── 2. 일별 집계 (LowTemp·AverageTemp·HeatIndex는 하루 단위 고정값) ──
    # 원본 df는 시간별이므로 일별 집계 후 다시 시간별로 merge
    daily = df.groupby(['Year', 'Month', 'Day']).agg(
        LowTemp_daily      = ('LowTemp',    'first'),   # 하루종일 동일
        AverageTemp_daily  = ('AverageTemp','first'),   # 평년 평균기온 근사
        MaxHeatIndex_daily = ('HeatIndex',  'max'),     # 일 최고 체감온도
    ).reset_index()

    # 전날 최저기온
    daily['LowTemp_prev1d']  = daily['LowTemp_daily'].shift(1)
    # 전날 대비 하강폭 (양수 = 하강)
    daily['LowTemp_drop_1d'] = daily['LowTemp_prev1d'] - daily['LowTemp_daily']
    # 평년 평균기온보다 3°C 이상 낮음
    daily['Below_avg3'] = (
        (daily['AverageTemp_daily'] - daily['LowTemp_daily']) >= 3
    ).astype(int)

    # ── 3. 한파 특보 조건 (10~4월만 적용) ───────────────────────────────
    cold_season = daily['Month'].isin([10, 11, 12, 1, 2, 3, 4])

    # 조건 ①: 전날 대비 최저기온 하강 + 3°C 이하 + 평년보다 3°C 낮음
    base_cond1 = (
        (daily['LowTemp_daily'] <= 3) &
        (daily['Below_avg3'] == 1)
    )
    cold_watch_c1  = cold_season & base_cond1 & (daily['LowTemp_drop_1d'] >= 10)
    cold_warn_c1   = cold_season & base_cond1 & (daily['LowTemp_drop_1d'] >= 15)

    # 조건 ②: 최저기온 -12°C / -15°C 이하 2일 이상 지속
    below_neg12 = (daily['LowTemp_daily'] <= -12).astype(int)
    below_neg15 = (daily['LowTemp_daily'] <= -15).astype(int)
    cold_watch_c2  = cold_season & (below_neg12.rolling(2, min_periods=2).min() == 1)
    cold_warn_c2   = cold_season & (below_neg15.rolling(2, min_periods=2).min() == 1)

    daily['Coldwave_Watch']   = (cold_watch_c1 | cold_watch_c2).astype(int)
    daily['Coldwave_Warning'] = (cold_warn_c1  | cold_warn_c2 ).astype(int)
    # 경보는 주의보보다 상위 → 경보이면 주의보도 1로 통일
    daily['Coldwave_Watch']   = np.maximum(daily['Coldwave_Watch'],
                                            daily['Coldwave_Warning'])
    # 순서 변수 (0=없음 / 1=주의보 / 2=경보)
    daily['Coldwave_Level'] = daily['Coldwave_Watch'] + daily['Coldwave_Warning']

    # ── 4. 폭염 특보 조건 ────────────────────────────────────────────────
    # 조건 ①: 일 최고 체감온도 33/35°C 이상 2일 이상 지속
    above_33 = (daily['MaxHeatIndex_daily'] >= 33).astype(int)
    above_35 = (daily['MaxHeatIndex_daily'] >= 35).astype(int)
    heat_watch_c1 = above_33.rolling(2, min_periods=2).min() == 1
    heat_warn_c1  = above_35.rolling(2, min_periods=2).min() == 1

    daily['Heatwave_Watch']   = heat_watch_c1.astype(int)
    daily['Heatwave_Warning'] = heat_warn_c1.astype(int)
    daily['Heatwave_Watch']   = np.maximum(daily['Heatwave_Watch'],
                                            daily['Heatwave_Warning'])
    daily['Heatwave_Level'] = daily['Heatwave_Watch'] + daily['Heatwave_Warning']

    # ── 5. 시간별 df에 일별 특보 변수 merge ──────────────────────────────
    merge_cols = [
        'Year', 'Month', 'Day',
        'LowTemp_prev1d', 'LowTemp_drop_1d', 'Below_avg3', 'MaxHeatIndex_daily',
        'Coldwave_Watch', 'Coldwave_Warning', 'Coldwave_Level',
        'Heatwave_Watch', 'Heatwave_Warning', 'Heatwave_Level',
    ]
    df = df.merge(daily[merge_cols], on=['Year', 'Month', 'Day'], how='left')

    # ── 6. 연속 특보 지속 시간 수 (시간 단위) ────────────────────────────
    # 특보가 해제된 뒤 다시 발령될 때 카운터 리셋
    df['Coldwave_streak'] = df.groupby(
        (df['Coldwave_Watch'] == 0).cumsum()
    )['Coldwave_Watch'].cumsum()

    df['Heatwave_streak'] = df.groupby(
        (df['Heatwave_Watch'] == 0).cumsum()
    )['Heatwave_Watch'].cumsum()

    # ── 검증 출력 ────────────────────────────────────────────────────────
    n_days = daily.shape[0]
    print("[기상청 기준 특보 변수 생성 완료]")
    print(f"  전체 {n_days}일 기준:")
    print(f"  한파 주의보: {daily['Coldwave_Watch'].sum():4d}일 | "
          f"한파 경보: {daily['Coldwave_Warning'].sum():4d}일")
    print(f"  폭염 주의보: {daily['Heatwave_Watch'].sum():4d}일 | "
          f"폭염 경보: {daily['Heatwave_Warning'].sum():4d}일")
    print(f"  추가 컬럼: {[c for c in df.columns if any(k in c for k in ['Wave','streak','HeatIndex','Below','drop','prev1d','Level'])]}")

    return df

In [ ]:
# ---------------------------------------------------------
# 2. 파생 변수 생성 함수
# ---------------------------------------------------------
def create_initial_features(df):
    df = df.copy()

    df['Datetime'] = pd.to_datetime(df[['Year', 'Month', 'Day', 'Hour']])
    df = df.sort_values('Datetime').reset_index(drop=True)

    T, H, V = df['Temp'], df['Humi'], df['Wind']
    df['DI']         = 0.81*T + 0.01*H*(0.99*T - 14.3) + 46.3
    df['Wind_Chill'] = 13.12 + 0.6215*T - 11.37*(V**0.16) + 0.3965*T*(V**0.16)
    df['Temp_Range'] = df['HighTemp'] - df['LowTemp']

    df['DayOfWeek']   = df['Datetime'].dt.dayofweek
    df['Is_Weekend']  = (df['DayOfWeek'] >= 5).astype(int)
    df['Is_Monday']   = (df['DayOfWeek'] == 0).astype(int)
    df['Is_Friday']   = (df['DayOfWeek'] == 4).astype(int)

    df['Hour_sin']    = np.sin(2*np.pi*df['Hour']/24)
    df['Hour_cos']    = np.cos(2*np.pi*df['Hour']/24)
    df['Month_sin']   = np.sin(2*np.pi*df['Month']/12)
    df['Month_cos']   = np.cos(2*np.pi*df['Month']/12)
    df['Day_sin']     = np.sin(2*np.pi*df['Day']/31)
    df['Day_cos']     = np.cos(2*np.pi*df['Day']/31)
    df['Weekday_sin'] = np.sin(2*np.pi*df['DayOfWeek']/7)
    df['Weekday_cos'] = np.cos(2*np.pi*df['DayOfWeek']/7)

    def get_season(m):
        if m in [3,4,5]: return 'Spring'
        elif m in [6,7,8]: return 'Summer'
        elif m in [9,10,11]: return 'Autumn'
        else: return 'Winter'
    df['Season'] = df['Month'].apply(get_season)

    def get_kepco_load_type(month, hour):
        if month in [6,7,8]:
            if 13<=hour<=17: return 'Peak'
            elif (9<=hour<=12) or (18<=hour<=22): return 'Mid'
            else: return 'Light'
        elif month in [11,12,1,2]:
            if (9<=hour<=12) or (16<=hour<=17): return 'Peak'
            elif (13<=hour<=15) or (18<=hour<=22): return 'Mid'
            else: return 'Light'
        else:
            if (9<=hour<=12) or (13<=hour<=17) or (18<=hour<=22): return 'Mid'
            else: return 'Light'
    df['Load_Type'] = df.apply(
        lambda r: get_kepco_load_type(r['Month'], r['Hour']), axis=1
    )

    kr_holidays = holidays.KR(years=[2015,2016,2017,2018])
    df['Is_Holiday'] = df['Datetime'].apply(lambda x: 1 if x in kr_holidays else 0)

    unique_dates  = df['Datetime'].dt.date.unique()
    rest_day_dict = {
        d: 1 if d.weekday()>=5 or d in kr_holidays else 0
        for d in unique_dates
    }

    df['Temp_Date']      = df['Datetime'].dt.date
    df.drop('Temp_Date', axis=1, inplace=True)
    df = add_kma_warning_features(df)

    df['Is_Jangma'] = (
        (df['Month'].isin([6,7])) & (df['Humi'] >= 80)
    ).astype(int)

    lags = [1, 24, 72, 168]
    for lag in lags:
        df[f'Temp_Lag{lag}']  = df['Temp'].shift(lag)
        df[f'DI_Lag{lag}']    = df['DI'].shift(lag)
        df[f'Humi_Lag{lag}']  = df['Humi'].shift(lag)

    windows = [6, 12, 24]
    for w in windows:
        df[f'Temp_RollMean_{w}'] = df['Temp'].rolling(w).mean()
        df[f'Temp_RollMax_{w}']  = df['Temp'].rolling(w).max()
        df[f'Temp_RollMin_{w}']  = df['Temp'].rolling(w).min()

    df = df.bfill()
    return df

# ---------------------------------------------------------
# 3. 전처리 파이프라인 실행
# ---------------------------------------------------------
print("\n파생 변수 생성을 시작합니다...")

train_features_step1 = create_initial_features(raw_train_df)
test_features_step1 = create_initial_features(raw_test_df)

print(f"전처리 완료 - Train: {train_features_step1.shape} | Test: {test_features_step1.shape}")

In [ ]:
# ---------------------------------------------------------
# 4. 데이터 스케일링 및 분할 (Train/Val)
# ---------------------------------------------------------
# 원본 보호를 위해 복사
train_data = train_features_step1.copy()
test_data = test_features_step1.copy()

train_data = pd.get_dummies(train_data, columns=['Season', 'Load_Type'])
test_data  = pd.get_dummies(test_data,  columns=['Season', 'Load_Type'])
test_data  = test_data.reindex(columns=train_data.columns, fill_value=0)

# 3. 불필요한 컬럼 제거
drop_cols = ['id', 'Datetime']
train_data.drop(columns=[c for c in drop_cols if c in train_data.columns], inplace=True)
test_data.drop( columns=[c for c in drop_cols if c in test_data.columns],  inplace=True)

n = len(train_data)
split_idx = int(n * 0.8)

train_df = train_data.iloc[:split_idx].copy()
val_df   = train_data.iloc[split_idx:].copy()
test_df  = test_data.copy()

print(f"Train 기간: {train_df['Year'].min()} ~ {train_df['Year'].max()} | {len(train_df)}행")
print(f"Val   기간: {val_df['Year'].min()}  ~ {val_df['Year'].max()}  | {len(val_df)}행")

# 4. 특성(X)과 타겟(y) 분리
y_train = train_df['UsedPower'].values.reshape(-1, 1)
X_train = train_df.drop(columns=['UsedPower']).values

y_val = val_df['UsedPower'].values.reshape(-1, 1)
X_val = val_df.drop(columns=['UsedPower']).values

X_test = test_df.drop(columns=['UsedPower'], errors='ignore').values

# 5. 스케일링 (Min-Max)
y_all = train_data['UsedPower'].values.reshape(-1, 1)

scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

X_train_scaled = scaler_X.fit_transform(X_train)
y_train_scaled = scaler_y.fit_transform(y_all)[:split_idx].flatten()

X_val_scaled   = scaler_X.transform(X_val)
y_val_scaled   = scaler_y.transform(y_val).flatten()

X_test_scaled  = scaler_X.transform(X_test)

print(f"\nscaler_y 학습 범위: {scaler_y.data_min_[0]:.1f} ~ {scaler_y.data_max_[0]:.1f}")
print(f"X_train: {X_train_scaled.shape} | X_val: {X_val_scaled.shape}")

In [ ]:
# Swish 활성화 함수 정의 (기존 ReLU가 음수 값을 완전히 무시해버리는 단점을 극복)
class Swish(nn.Module):
    def forward(self, x):
        return x * torch.sigmoid(x)


def get_activation(name: str) -> nn.Module:
    """
    활성화 함수 이름 → nn.Module 반환.
    Optuna trial.suggest_categorical 결과값을 그대로 넘기면 됩니다.
    """
    return {
        'Swish'     : Swish(),
        'LeakyReLU' : nn.LeakyReLU(negative_slope=0.1),
        'PReLU'     : nn.PReLU(),
    }[name]

class ImprovedHybridModel(nn.Module):
  def __init__(self, input_dim: int, hidden_dim: int,
                activation_fn: nn.Module,
                num_layers: int = 2,
                dropout: float = 0.2):
      super().__init__()

      # 1D Conv: 지역 패턴(단기 주기) 추출
      self.conv1d     = nn.Conv1d(input_dim, hidden_dim, kernel_size=3, padding=1)
      self.activation = activation_fn
      self.bn_conv    = nn.BatchNorm1d(hidden_dim)

      # 다층 GRU
      self.gru = nn.GRU(
          hidden_dim, hidden_dim,
          num_layers=num_layers,
          batch_first=True,
          dropout=dropout if num_layers > 1 else 0.0,
      )

      # ── Temporal Attention ──────────────────────────────
      # hidden_dim → 1 스칼라 점수를 각 시점마다 계산 → softmax → 가중 합
      self.temporal_attn  = nn.Linear(hidden_dim, 1)

      # Feature Attention
      self.feature_gate   = nn.Linear(hidden_dim, hidden_dim)

      # Dropout
      self.dropout        = nn.Dropout(dropout)

      # 출력 레이어
      self.fc             = nn.Linear(hidden_dim, 1)

  def forward(self, x):
      # x : (B, T, F)
      B, T, F = x.shape

      # ── Conv1d 블록 ────────────────────────────────────────────────
      # permute: (B, T, F) → (B, F, T)
      h = self.activation(self.conv1d(x.permute(0, 2, 1)))  # (B, H, T)
      h = self.bn_conv(h)
      h = h.permute(0, 2, 1)                                  # (B, T, H)

      # ── GRU ───────────────────────────────────────────────────────
      out_gru, _ = self.gru(h)                                # (B, T, H)

      # ── Temporal Attention ──────────────────────
      # 각 시점마다 스칼라 score 계산 → softmax → context vector
      attn_scores  = self.temporal_attn(out_gru)              # (B, T, 1)
      attn_weights = torch.softmax(attn_scores, dim=1)        # (B, T, 1)  ← 시간 축 softmax
      context      = (out_gru * attn_weights).sum(dim=1)      # (B, H)     ← 가중 합

      # ── Feature Gate ───────────────────────────────────────────────
      gate    = torch.sigmoid(self.feature_gate(context))     # (B, H)
      gated   = context * gate                                 # (B, H)

      # ── 출력 ────────────────────────────────────────────────────────
      out = self.fc(self.dropout(gated)).squeeze(-1)           # (B,)
      return out

In [ ]:
# ──────────────────────────────────────────────
# 3. 손실 함수
# ──────────────────────────────────────────────
class ScaledHuberLoss(nn.Module):
  def __init__(self, delta: float = 0.1):
      super().__init__()
      self.huber = nn.HuberLoss(delta=delta)

  def forward(self, pred, true):
      return self.huber(pred, true)

In [ ]:
# 시계열 데이터셋 클래스 구축
class TimeSeriesDataset(Dataset):
    def __init__(self, features, target, window_size=24):
        """
        features: 스케일링이 완료된 2차원 numpy 배열 (독립변수)
        target: 스케일링이 완료된 1차원 numpy 배열 (종속변수, UsedPower)
        window_size: 모델이 한 번에 바라볼 과거 데이터의 시간 길이 (예: 24시간)
        """
        self.features = features
        self.target = target
        self.window_size = window_size

    def __len__(self):
        # 전체 데이터 길이에서 윈도우 사이즈를 뺀 만큼 샘플 생성 가능
        return len(self.features) - self.window_size

    def __getitem__(self, idx):
        # idx부터 idx + window_size 전까지의 데이터를 X로 구성
        x_window = self.features[idx : idx + self.window_size]
        # window_size 시점의 정답 데이터를 y로 구성
        y_target = self.target[idx + self.window_size]

        # PyTorch Tensor로 변환
        x_tensor = torch.tensor(x_window, dtype=torch.float32)
        y_tensor = torch.tensor(y_target, dtype=torch.float32)

        return x_tensor, y_tensor

In [ ]:
def get_dataloaders(X_train, y_train, X_val, y_val, window_size, batch_size):
    train_ds = TimeSeriesDataset(X_train, y_train, window_size)
    val_ds   = TimeSeriesDataset(X_val, y_val, window_size)

    g = torch.Generator(); g.manual_seed(42)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, generator=g)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False)

    return train_loader, val_loader

In [ ]:
# ──────────────────────────────────────────────
# 5. Optuna Objective
# ──────────────────────────────────────────────
def objective(trial, X_train_scaled, y_train_scaled,
              X_val_scaled, y_val_scaled, scaler_y):

    window_size     = trial.suggest_categorical('window_size', [24, 48, 72, 168])
    hidden_dim      = trial.suggest_categorical('hidden_dim',  [64, 128, 256])
    lr              = trial.suggest_float('lr', 1e-4, 5e-3, log=True)
    batch_size      = trial.suggest_categorical('batch_size',  [32, 64, 128])
    activation_name = trial.suggest_categorical('activation',  ['Swish', 'LeakyReLU', 'PReLU'])
    dropout         = trial.suggest_float('dropout', 0.1, 0.4)
    num_layers      = trial.suggest_categorical('num_layers', [1, 2])
    huber_delta     = trial.suggest_float('huber_delta', 0.05, 0.3)

    train_loader, val_loader = get_dataloaders(
        X_train_scaled, y_train_scaled,
        X_val_scaled, y_val_scaled,
        window_size, batch_size
    )

    model = ImprovedHybridModel(
        input_dim     = X_train_scaled.shape[1],
        hidden_dim    = hidden_dim,
        activation_fn = get_activation(activation_name),
        num_layers    = num_layers,
        dropout       = dropout,
    ).to(device)

    criterion = ScaledHuberLoss(delta=huber_delta)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    # LR Scheduler: 성능 정체 시 학습률 감소
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=3
    )

    for epoch in range(20):
        model.train()
        for bx, by in train_loader:
            bx, by = bx.to(device), by.to(device)
            optimizer.zero_grad()
            loss = criterion(model(bx), by)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # Gradient Clipping
            optimizer.step()

        model.eval()
        val_preds, val_trues = [], []
        with torch.no_grad():
            for vx, vy in val_loader:
                val_preds.extend(model(vx.to(device)).cpu().numpy())
                val_trues.extend(vy.cpu().numpy())

        # 역정규화 후 MAPE 계산
        vp_mw = scaler_y.inverse_transform(np.array(val_preds).reshape(-1,1))
        vt_mw = scaler_y.inverse_transform(np.array(val_trues).reshape(-1,1))
        mape  = mean_absolute_percentage_error(vt_mw, vp_mw)

        scheduler.step(mape)
        trial.report(mape, epoch)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return mape

In [ ]:
def train_final_model(best_params, X_train_scaled, y_train_scaled,
                      X_val_scaled, y_val_scaled, scaler_y,
                      final_epochs=100, patience=15, loss_fn=None):
  bw  = best_params['window_size']
  bh  = best_params['hidden_dim']
  blr = best_params['lr']
  bb  = best_params['batch_size']
  ban = best_params['activation']
  bd  = best_params['dropout']
  bnl = best_params['num_layers']
  bdelta = best_params['huber_delta']

  train_ds = TimeSeriesDataset(X_train_scaled, y_train_scaled, bw)
  val_ds   = TimeSeriesDataset(X_val_scaled,   y_val_scaled,   bw)

  g = torch.Generator(); g.manual_seed(42)
  train_loader = DataLoader(train_ds, batch_size=bb, shuffle=True, generator=g)
  val_loader   = DataLoader(val_ds,   batch_size=bb, shuffle=False)

  model = ImprovedHybridModel(
      input_dim     = X_train_scaled.shape[1],
      hidden_dim    = bh,
      activation_fn = get_activation(ban),
      num_layers    = bnl,
      dropout       = bd,
  ).to(device)

  criterion = loss_fn if loss_fn is not None else ScaledHuberLoss(delta=bdelta)
  optimizer = torch.optim.Adam(model.parameters(), lr=blr)
  scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
      optimizer, mode='min', factor=0.5, patience=5)

  best_mape     = float('inf')
  best_wts      = copy.deepcopy(model.state_dict())
  no_improve    = 0

  for epoch in range(final_epochs):
      # ── 학습 ──
      model.train()
      train_loss = 0.0
      for bx, by in train_loader:
          bx, by = bx.to(device), by.to(device)
          optimizer.zero_grad()
          loss = criterion(model(bx), by)
          loss.backward()
          nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
          optimizer.step()
          train_loss += loss.item()

      # ── 검증 ──
      model.eval()
      val_preds, val_trues = [], []
      with torch.no_grad():
          for vx, vy in val_loader:
              val_preds.extend(model(vx.to(device)).cpu().numpy())
              val_trues.extend(vy.cpu().numpy())

      vp_mw = scaler_y.inverse_transform(np.array(val_preds).reshape(-1,1))
      vt_mw = scaler_y.inverse_transform(np.array(val_trues).reshape(-1,1))
      mape  = mean_absolute_percentage_error(vt_mw, vp_mw)

      scheduler.step(mape)

      if (epoch + 1) % 10 == 0 or epoch == 0:
          print(f"  Epoch [{epoch+1:3d}/{final_epochs}] | "
                f"Train Loss: {train_loss/len(train_loader):.5f} | "
                f"Val MAPE: {mape*100:.2f}%")

      # ── Early Stopping ──
      if mape < best_mape:
          best_mape  = mape
          best_wts   = copy.deepcopy(model.state_dict())
          no_improve = 0
      else:
          no_improve += 1
          if no_improve >= patience:
              print(f"\n⏹ Early Stopping at epoch {epoch+1} (patience={patience})")
              break

  model.load_state_dict(best_wts)
  print(f"\n✅ 최종 Validation MAPE: {best_mape*100:.2f}%")
  return model, best_mape

In [ ]:
def predict_full_for_submission(model, features_scaled, window_size, batch_size=256):
  model.eval()
  preds   = []
  n       = len(features_scaled)
  dataset = TimeSeriesDataset(features_scaled, np.zeros(n), window_size=window_size)
  loader  = DataLoader(dataset, batch_size=batch_size, shuffle=False)

  with torch.no_grad():
      for bx, _ in loader:
          preds.extend(model(bx.to(device)).cpu().numpy())

  pad        = [preds[0]] * window_size
  preds_full = np.array(pad + preds)
  return preds_full

In [ ]:
study = optuna.create_study(
    direction='minimize',
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(),
)
study.optimize(
    lambda trial: objective(
        trial,
        X_train_scaled, y_train_scaled,
        X_val_scaled,   y_val_scaled,
        scaler_y
    ),
    n_trials=50
)
print("최적 파라미터:", study.best_params)
print(f"최저 Validation MAPE: {study.best_value*100:.2f}%")

# 최종 학습
best_model, best_mape = train_final_model(
    best_params       = study.best_params,
    X_train_scaled    = X_train_scaled,
    y_train_scaled    = y_train_scaled,
    X_val_scaled      = X_val_scaled,
    y_val_scaled      = y_val_scaled,
    scaler_y          = scaler_y,
    final_epochs      = 100,
    patience          = 15,
)

# 테스트 추론 (기존과 동일)
best_window       = study.best_params['window_size']
test_preds_scaled = predict_full_for_submission(best_model, X_test_scaled, best_window)
test_preds_mw     = scaler_y.inverse_transform(test_preds_scaled.reshape(-1,1)).flatten()

In [ ]:
# ==========================
# 최종 모델 예측 수행
# ==========================
best_model.eval()

val_preds, val_trues = [], []

with torch.no_grad():
    for val_x, val_y in val_loader_best:
        pred = best_model(val_x.to(device))

        val_preds.extend(pred.cpu().numpy())
        val_trues.extend(val_y.cpu().numpy())

# 역정규화
val_preds_mw = scaler_y.inverse_transform(
    np.array(val_preds).reshape(-1, 1)
).flatten()

val_trues_mw = scaler_y.inverse_transform(
    np.array(val_trues).reshape(-1, 1)
).flatten()


# ==========================
# 실제값 vs 예측값 그래프
# ==========================
import matplotlib.pyplot as plt

plt.figure(figsize=(15, 6))

# 실제값
plt.plot(
    val_trues_mw,
    label='Actual',
    color='gray',
    alpha=0.5,
    linewidth=2
)

# 예측값
plt.plot(
    val_preds_mw,
    label='Prediction',
    color='tab:blue',
    linewidth=2.5,
    alpha=0.8
)

plt.title('Validation Set: Actual vs Prediction')
plt.xlabel('Time Step')
plt.ylabel('Power Load (MW)')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()